# Intro to GeoPandas and GeoDataFrames

A **GeoDataFrame** is a pandas DataFrame with an extra feature: a `geometry` column that allows us to plot, measure, and do other spatial operations

Today's data: Moving on from Ohio (for a bit), looking at examples from the Chesapeake Bay Watershed

By the end of this session you will be able to:

- Read a GeoDataFrame from a URL with `gpd.read_file()` and inspect its `.crs`.
- Reproject a GeoDataFrame with `.to_crs()`.
- Filter rows with pandas boolean syntax and compare it to a week-6 SQL where-clause

## Setup

Same as the last few times

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

## new data: `cbw_counties.geojson`

`DATA_BASE_P2` is Part 2's own data package. This is distinct from `DATA_BASE`, which was the constant that pointed at Part 1's continuity files.

***Most weeks from here on define `DATA_BASE_P2` and load straight from it***

`cbw_counties.geojson` covers seven states' worth of counties around the Chesapeake Bay watershed. 


In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2" ## This is different

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import geopandas as gpd

counties = gpd.read_file(f"{DATA_BASE_P2}/cbw_counties.geojson")
counties.head()

## What do we get?

- a table that looks exactly like a pandas DataFrame — `state_fp`, `county_fp`, `geoid`, `name`, `name_long`, `state`
- a column that isn't like the others: `geometry`, holding a shapely `Polygon` object per row 

***The structure of a GeoDataFrame: table + geometry***

...everything you already know about pandas (`.head()`, `.shape`, boolean filtering, `.groupby()`) still applies



## What makes it "Geo"?

Three things a plain pandas DataFrame doesn't have:

1. `.crs` (the coordinate reference system)
2. `.geom_type` (what kind of geometry each row holds)
3. and a working `.plot()` function with no additional extra setup

In [ ]:
print(counties.crs)
print(counties.geom_type.value_counts())

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
counties.plot(ax=ax, edgecolor="white", linewidth=0.3, color="steelblue")
ax.set_title("Chesapeake Bay watershed counties (cbw_counties.geojson)")
plt.show()

***What do we see?***

**Your turn:** Load `cbw_stations.geojson` into a variable called `stations`, and print its `.crs` and `.geom_type.value_counts()`


In [ ]:
## Try it here

## Reprojecting spatial data using `.to_crs()`

A GeoDataFrame's `.crs` describres the CRS the geometry column is expressed in

`.to_crs()` recomputes every single coordinate into a different reference system and returns a new GeoDataFrame; 

*it does not modify `counties` in place*

**What are the implications of that design choice?**

**Predict before you run:** `counties` is currently in EPSG:4269 (degrees). `EPSG:26918` is NAD83 / UTM zone 18N — a **projected** CRS in meters, appropriate for the Chesapeake region. 

Predict: will `.total_bounds` (`[minx, miny, maxx, maxy]`) after reprojecting still look like small numbers in the 30s and 70s (degrees), or something in the hundreds of thousands to millions (meters)?

In [ ]:
print("EPSG:4269 bounds:", counties.total_bounds)

counties_utm = counties.to_crs(26918)
print("EPSG:26918 bounds:", counties_utm.total_bounds)
print(counties_utm.crs)

### Result???

In [ ]:
# A second worked example: Web Mercator, which is the CRS almost every web basemap uses
counties_webmerc = counties.to_crs(3857)
print("EPSG:3857 bounds:", counties_webmerc.total_bounds)
print(counties_webmerc.crs.is_geographic, counties_webmerc.crs.is_projected)

**Expected result:** Web Mercator (EPSG:3857) is also projected (meters), but it uses a different false origin and a different (and famously area-distorting) projection than UTM. `is_geographic` reads `False`, `is_projected` reads `True` for both reprojected versions — a CRS is one or the other, never both, and programmatically checking that pair of booleans in code beats eyeballing whether numbers "look like degrees."

**Your turn:** Reproject `counties` to `EPSG:5070` (NAD83 / Conus Albers, the USGS's standard equal-area projection for the contiguous US) into a variable `counties_albers`. 

Then print its `.total_bounds` and confirm `counties_albers.crs.is_projected` is `True`. 



### Let's take a second: any questions so far?

## Attribute filtering in pandas

In ArcPy, we used `SelectLayerByAttribute` where-clauses like `'"ALAND" > 10000000'` and `"\"NAME\" = 'Kent'"` 

The field name was double-quoted, text value single-quoted, numeric value bare. Hard to keep straight

A GeoDataFrame skips the string-building entirely

### Let's walk through the syntax

In [ ]:
virginia = counties[counties["state"] == "VA"]
print(f"{len(virginia)} of {len(counties)} counties are in Virginia")
virginia[["name", "state"]].head()

***Results???***

How is this different than the "arcpy way"?

In [ ]:
# A second worked example: text-pattern filtering
independent_cities = counties[~counties["name_long"].str.contains("County", na=False)] ### THIS IS NEW - LET'S TAKE A LOOK
print(f"{len(independent_cities)} units are NOT a '...County' — independent cities and DC")
independent_cities[["name", "name_long", "state"]].head(8)

pandas' `.str.contains()` (here, negated with `~` for "does *not* contain") is a way to do a filter based on a string-matching rule

***How is `.str.contains()` different than `==` ???***

**Your turn:** Filter `counties` down to Pennsylvania (`state == "PA"`) into a variable `pennsylvania`, and print how many rows it has. 

Then try a second condition: how many counties have a `name` that is exactly 4 characters long (hint: `.str.len()`)?


In [ ]:
## Try it here:



## The ArcPy cursor, as one line on pandas code

A few weeks ago when using arcpy, I promised a version of: *"Starting in Part 2, once you're working in pandas and geopandas, this entire cell - open a cursor, loop it, print two fields - collapses into one line: `gdf[["NAME", "geometry"]].head()`."* 


Let's see if I was telling the truth :)

### Flashback to arcpy cursors (sorry)

```python
with arcpy.da.SearchCursor("ohio_counties.shp", ["NAME", "SHAPE@AREA"]) as cur:
    for i, row in enumerate(cur):
        if i >= 5:
            break
        print(row)
```

This printed 5 tuples in the shapefile's internal row order: `Hancock`, `Allen`, `Lake`, `Fayette`, `Butler`.

In [ ]:
# Same 88 Ohio counties as week 4's shapefile, now stored as a GeoPackage,
# read_file does not care (too much) about the file format, but the
# layer= argument isn't strictly required because this GeoPackage only contains a single layer
ohio = gpd.read_file(f"{DATA_BASE_P2}/ohio_counties.gpkg")
ohio[["NAME", "geometry"]].head()

**Expected result:** `Hancock`, `Allen`, `Lake`, `Fayette`, `Butler`... just has we had with arcpy

**Predict before you run:** Week 4's cursor also used `SHAPE@AREA` syntax, though we ended up with **square degrees**


Geopandas`gdf.area` will fall into that exact same trap (it's just arithmetic on whatever coordinates the geometry column currently holds). 

Guess: will ranking Ohio's counties by `gdf.area` right now, *before* any `.to_crs()` call, give you a geographically meaningful "5 biggest counties" answer?

In [ ]:
ohio_ranked_deg = ohio.assign(area_deg2=ohio.geometry.area).nlargest(5, "area_deg2")
ohio_ranked_deg[["NAME", "area_deg2"]]

**Expected result:** a `UserWarning` — "Geometry is in a geographic CRS. Results from 'area' are likely incorrect." — then a table topped by Ashtabula, Cuyahoga, Lake, Lorain, and Licking, with `area_deg2` values around `0.15`–`0.38`. 

GeoPandas is telling you what you already know: square degrees aren't a real unit of area, and a county's east-west degree of longitude covers less ground than a degree of latitude at this latitude.

So what? This ranking is *arithmetically* correct but *geographically* suspect. 

Don't trust an area number from a geographic CRS just because a warning didn't stop the code from running!!!

### So what's the solution???

In [ ]:
# Reproject first, exactly like the section above taught, then rank again
ohio_utm = ohio.to_crs(26917)  # NAD83 / UTM zone 17N — meters, appropriate for Ohio
ohio_ranked_km2 = ohio_utm.assign(area_km2=ohio_utm.geometry.area / 1e6).nlargest(5, "area_km2")
ohio_ranked_km2[["NAME", "area_km2"]]

**Expected result:** Ashtabula, Cuyahoga, Lake, and Lorain still lead, in the same order... but the #5 spot changes: 

Ross County (real area ~1,795 km²) displaces Licking County, which ranked #5 in the uncorrected degrees version but drops out of the real top 5 once area is computed in meters. 

This is why we reproject into propr CRS.

But also, one line replaced the arcpy cursor block for reading rows. One more `.to_crs()` call replaced the geometry-token unit trap

**Your turn:** Using `ohio_utm` (already reprojected to meters), find Ohio's 5 *smallest* counties by area with `.nsmallest(5, "area_km2")` instead of `.nlargest(...)` (you'll need to `.assign()` the `area_km2` column again, or reuse the one already on `ohio_ranked_km2` if you kept every row instead of just the top 5). 

Are any of the same five counties from today's `virginia`/`independent_cities` filters recognizable place names, or is this a genuinely different part of the file?

## If there's time

**Try it yourself**:

1. Load `cbw_stations.geojson` and filter it down to `station_type == "trib"` (tributary stations, as opposed to `"main"` mainstem-Bay stations). How many rows?
2. Reproject `ohio` to `EPSG:4326` (WGS84) instead of `EPSG:26917`, and compare `ohio.crs` and `ohio_wgs84.crs` — they're both geographic CRSs, but not the same one. Would ranking by area still be safe here? (Predict, then check the warning again if you try it.)
3. Pick any column in `counties` other than `geometry` and run `.value_counts()` on it — get a feel for what a real Chesapeake attribute table looks like before Thursday's session starts joining and dissolving it.



## Next class

1. the rest of the vector toolbox (buffer, spatial join, dissolve, overlay)

2. Lab 4 introduction